# Lab Assignment: Generative AI

#### Lab team: teamCode

##### Name (member 1): Jimena de Prado González

* Please include your full name at the beginning of all submitted files.
* Make sure the presentation is well-structured: the report will be evaluated not only for correctness, but also for clarity, conciseness, and completeness.
* Make use of figures and tables to summarize the results and illustrate the discussions.
* If external material is used, the sources must be cited. 
* Include references in APA format https://pitt.libguides.com/citationhelp/apa7. Lack or poorly formatted references can be penalized.
* A generative AI tool can be used for consultation. You must specify the tool used in your report.
* You are not allowed to use a generative AI tool to generate code.

Submit a single `.zip` file, whose name has the format `AA3_2024_2025_P02_teamCode_lastName1_lastName2.zip` 
The name must not include graphical accents, spaces, uppercase letters, or special characters.

For example: `AA3_2024_2025_P02_V03_munyoz_deLaRosa.zip`

This compressed file must include the following files:

* This Python notebook with the solutions of the exercises. The notebook should include only code snippets, figures, tables, derviations and explanations (with LaTex if necessary) in Markdown cells. Handwritten material can be included in the Python notebook as images. Functions should be defined in a separate `.py` file, not in the notebook.
* The necessary `.py` and addional files to ensure the Python notebook code can be executed sequentially without errors.
* A PDF file generated from the notebook (Export the notebook as an HTML file. Open the HTML file in a Browser and print it as a PDF file). 

Make sure that all the code cells can be executed squentially without errors (Kernel -> Restart & Run All). Exectution and formatting errors will be penalized.

The grade of this lab assignment is based on
* This submission (50 %).
* An individual in-class exam (50%).


Evaluation criteria:
* [6 points] Quality of the report (correctness, clarity, conciseness, completeness).
* [3 points] Quality of the code (correctness, adherence to a Python style guide -for instance, Google's-, comments, functional decomposition).
* [1 point]  References.                                                                   

## References:

1. Yang Song, Jascha Sohl-Dickstein, Diederik P. Kingma, Abhishek Kumar, Stefano Ermon and Ben Poole
"Score-Based Generative Modeling through Stochastic Differential Equations"
In International Conference on Learning Representations, 2021
https://arxiv.org/abs/2011.13456
2. TODO: Include references in alphabetical order

In [6]:
%load_ext autoreload
%autoreload 2

from score_model import ScoreNet
import diffusion_process as dfp
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, Subset
import torch
from diffusion_utilities import plot_image_evolution

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Exercise 1:  Generative AI based on diffusion models: Brownian motion 

In this exercise the training data is generated by injectiong noise using Brownian motion (variance exploding diffusion model).
The stochastic differencial equation (SDE) that characterizes this forward diffusion process is
$$
d\mathbf{x}(t) = \sigma^t d\mathbf{W}(t) 
$$

with $ \mathbf{x}(0) \sim p_0\left(\mathbf{x} \right)$, and  $\sigma > 0 $.
1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary t?
2. What is the SDE for the time-reversed process? 
3. For the synthesis of new images
    1. In what interval is the reverse process SDE integrated?
    2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled?

### Exercise 1.1:  Training the model by weighted sum of denoising score matching objectives
#### 1. Give the mathematical form and explain the cost function used in the training of the generative AI model. 

$$
\mathcal{L}(\theta) =
\mathbb{E}_{t \sim \mathcal{U}(\epsilon,1),\; x_0 \sim p_{\text{data}},\; z \sim \mathcal{N}(0,I)}
\left[
\left\|
\sigma(t)\, s_\theta(x_t, t) + z
\right\|^2
\right]
$$

$$
\text{where} \quad x_t = x_0 + \sigma(t)\, z
$$

The cost function is based on the idea of denoising score matching. Starting from a clean data sample $x_0$, Gaussian noise $z \sim \mathcal{N}(0,I)$ is added to obtain a perturbed sample $x_t$ at noise level $t$.

For this Gaussian perturbation, the score (i.e., the gradient of the log-density) is known analytically:

$$
\nabla_x \log p(x_t \mid x_0) = -\frac{1}{\sigma(t)} z
$$

The neural network $s_\theta(x_t, t)$ is trained to approximate this score. Therefore, the loss function is defined as the mean squared error between the predicted score and the true score:

$$
\mathcal{L}(\theta) =
\mathbb{E}
\left[
\left\|
s_\theta(x_t,t) + \frac{1}{\sigma(t)} z
\right\|^2
\right]
$$

Multiplying by $\sigma(t)$ leads to a stable formulation:

$$
\mathcal{L}(\theta) =
\mathbb{E}
\left[
\left\|
\sigma(t)\, s_\theta(x_t,t) + z
\right\|^2
\right]
$$

This objective can be interpreted as a weighted denoising problem: the model learns to predict the noise added to the data at different noise levels, with the weighting controlled by $\sigma(t)$.

#### 2. Indicate what type of neural network is used to model the score, its inputs, outputs, and architecture.   

- The score is modeled using a **deep convolutional neural network** with a U-Net architecture.

- **Inputs:** it takes as input a noisy image $x_t$ and a time variable $t$ representing the noise level. The scalar $t$ is mapped into a higher-dimensional representation using Gaussian random Fourier features.

- **Output:** it outputs $s_\theta(x_t, t)$, which approximates the score function $\nabla_x \log p(x_t)$ and has the same dimensionality as the input image.

- **Architecture**(U-Net structure):

    - An encoder (downsampling path) that extracts hierarchical features using convolutional layers  
    - A decoder (upsampling path) that reconstructs spatial resolution using transposed convolutions  
    - Skip connections that link encoder and decoder layers to preserve spatial information  
    - Time conditioning, where the embedding of $t$ is injected into intermediate layers  
    - A final normalization by $\sigma(t)$ to match the theoretical scaling of the score 

- The network learns to estimate the score of the noisy data distribution, which provides the direction of higher probability density and allows reversing the diffusion process.

#### 3. Explain how time is input in the neural network model for the time-dependent score function using random Fourier features.

The time variable $t$ is incorporated into the neural network using a high-dimensional embedding based on Gaussian random Fourier features. It is transformed into a more broad representation that the network can process more effectively, since a single scalar does not provide enough power.

First, a set of random frequencies is sampled from a Gaussian distribution:

$$
w_i \sim \mathcal{N}(0, \text{scale}^2)
$$

Then, the time variable $t$ is mapped into a higher-dimensional vector using sinusoidal functions:

$$
\phi(t) =
\left[
\sin(2\pi w_1 t), \cos(2\pi w_1 t), \dots,
\sin(2\pi w_k t), \cos(2\pi w_k t)
\right]
$$

This produces an embedding $\phi(t)$ that encodes the time variable at multiple frequencies.

The resulting embedding is then passed into the neural network, typically by adding it to intermediate feature maps. This allows the model to adapt its behavior depending on the noise level.

#### 4. Illustrate how to train a model to generate handwritten digits for the MNIST dataset using the Brownian motion diffusion model. This first block of code should run in less than 5 minutes.

To train a model that generates handwritten digits using the MNIST dataset and a Brownian motion (variance exploding) diffusion model, we follow a denoising score matching approach. First, clean images $x_0$ are sampled from the dataset. Then, Gaussian noise is added to obtain noisy samples:

$$
x_t = x_0 + \sigma(t) z, \quad z \sim \mathcal{N}(0, I)
$$

A neural network is trained to estimate the score function:

$$
s_\theta(x_t, t) \approx \nabla_x \log p(x_t)
$$

The training objective is to minimize the following loss function:

$$
\mathcal{L}(\theta) =
\mathbb{E}
\left[
\left\|
\sigma(t)\, s_\theta(x_t, t) + z
\right\|^2
\right]
$$

During training, for each image in the dataset:

- A random time $t$ is sampled  
- Noise $z$ is generated  
- A noisy image $x_t$ is constructed  
- The neural network predicts the score $s_\theta(x_t, t)$  
- The prediction is compared to the true noise  

By minimizing this loss, the model learns to predict the noise added to the data. This allows the model to later reverse the diffusion process and generate new handwritten digits starting from pure noise. After training, the model can generate new samples by reversing the diffusion process.

Starting from pure noise:

$$
x_T \sim \mathcal{N}(0, I)
$$

the model iteratively updates the sample using the learned score function:

$$
dx_t = -g(t)^2 s_\theta(x_t, t)\, dt + g(t)\, d\bar{W}_t
$$

where $g(t)$ is the diffusion coefficient.

At each step, the model uses the estimated score to move the sample toward regions of higher data probability, gradually removing noise.

In [5]:
device = 'cpu'
batch_size = 32

data = datasets.MNIST(root="data", train=True, download=True, transform=ToTensor())

digit = 5
indices = torch.where(data.targets == digit)[0]
data_train = Subset(data, indices)

data_loader = DataLoader(data_train, batch_size=batch_size, shuffle=True)

# Diffusion model (Brownian motion)
sigma = 25.0

def drift(x_t, t):
    return torch.zeros_like(x_t)

def diffusion(t):
    return sigma ** t

def mu_t(x_0, t):
    return x_0

def sigma_t(t):
    return torch.sqrt(0.5 * (sigma**(2*t) - 1.0) / torch.log(torch.tensor(sigma)))

diffusion_process = dfp.GaussianDiffussionProcess(
    drift, diffusion, mu_t, sigma_t
)

#Score model
score_model = ScoreNet(marginal_prob_std=sigma_t)
score_model = score_model.to(device)

#Optimizer
optimizer = torch.optim.Adam(score_model.parameters(), lr=1e-3)

for epoch in range(3): 
    for x, _ in data_loader:
        x = x.to(device)

        loss = diffusion_process.loss_function(score_model, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 135.3988
Epoch 1, Loss: 66.7680
Epoch 2, Loss: 66.6714


### Exercise 1.2: Generation using the Euler-Maruyama integrator
Use a model that you have previously trained (not the one in the previous exercise) high-quality model to generate some samples using the Euler-Maruyama integrator.

### Exercise 1.3: Generation using a probability flow ODE.
1. Explain what is the Fokker-Planck equation and in what way is it related to an SDE.
2. Explain how the probability flow ODE can be used to generate samples from  $p_0\left(\mathbf{x} \right)$.
2. Implement and illustrate the use of this method to generate synthetic images of handwritten digits. 
3. Indicate how to use this scheme to compute likelihoods. Implement this functionality and illustrate its use.

## Exercise 2:  Generative AI based on diffusion models: The Ornstein-Uhlenbeck process

In this exercise, the training data is generated by injecting noise using Brownian motion (variance preserving diffusion model).
The stochastic differencial equation (SDE) that characterizes this forward diffusion process is
$$
d\mathbf{x}(t) = - \frac{1}{2} \beta(t) \mathbf{x}(t) + \sqrt{\beta(t)} d\mathbf{W}(t) 
$$
with $ \mathbf{x}(0) \sim p_0\left(\mathbf{x} \right)$
1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary t?
2. What is the SDE for the time-reversed process? 
3. For the synthesis of new images
        1. In what interval is the reverse process SDE integrated?
        2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled? 

### Exercise 2.1: Training and generation of images using the OU process and different noise schedules
1. Using the linear noise schedule.
2. Using the cosine noise schedule.
3. Using a third noise schedule of your choice.

## Exercise 3:  Evaluation of the quality of the generated images
1. Describe, and compare the characteristics, advantages, and disadvantages of the following measures of quality for generative IA models for images:
    1. The negative log-likelihood (NLL).
    2. Bits per dimension (BPD). 
    3. Fréchet Inception Distance (FID).
    4. A third measure of your choice.
2. Compare using BPD the different diffusion models (Brownian motion, Ornstein-Uhlenbeck), noising schedules (linear, cosine, etc.), and sampling strategies (SDE, ODE) implemented.